In [ ]:
%pip install rasterio
%pip install numpy
%pip install pandas
%pip install plotly
%pip install geopandas
%pip install shapely

Import Dependencies

In [ ]:
import plotly.graph_objects as go
import numpy as np
import rasterio
from rasterio.windows import from_bounds
import rasterio.transform
from rasterio.plot import show
import json
from shapely.geometry import shape
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
tiff = rasterio.open(r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_aug.tif')
show(tiff, title="Raster Visualization")

# Adding a US States Shapefile to the project

#### Importing a shapefile

Importing State Boundary Shape Files & Converting to WGS84 Projection

In [ ]:
shapefile_path = r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\tl_2024_us_state\tl_2024_us_state.shp'

# Load the U.S. states shapefile
gdf = gpd.read_file(shapefile_path)

# Reproject to WGS84 (latitude/longitude)
gdf = gdf.to_crs("EPSG:4326")

Obtaining desired bonuds & filtering state boundary shape file to match Raster Images.

In [ ]:
# Define bounds explicitly
DESIRED_BOUNDS = {
    'left': -140,
    'right': -40,
    'bottom': 10,
    'top': 60
}

from shapely.geometry import box

# Define raster bounding box
raster_bounds = box(DESIRED_BOUNDS['left'], DESIRED_BOUNDS['bottom'],
                     DESIRED_BOUNDS['right'], DESIRED_BOUNDS['top'])

# Clip the GeoDataFrame to the bounding box
filtered_gdf = gdf.clip(raster_bounds)

# Verify by plotting
filtered_gdf.plot(edgecolor="black", facecolor="none")
plt.title("Filtered U.S. State Boundaries (Clipped)")
plt.show()

# Convert to GeoJSON for Plotly
filtered_geojson = filtered_gdf.to_json()


##### Adding the Raster File Paths

In [ ]:
# Configure Paths to the Raster Files
RASTER_PATHS = [
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_jan.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_feb.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_mar.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_apr.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_may.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_jun.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_jul.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_aug.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_sep.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_oct.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_nov.tif',
    r'C:\Users\runni\OneDrive\Documents\UT\DViz Project\data\nsrdbv3_dni\nsrdbv3_dni\Monthly DNI\dni_dec.tif'
]

#### Precomputing & Loading Raster Files

In [ ]:
# Precompute coordinates from the first file
with rasterio.open(RASTER_PATHS[0]) as src:
    window = from_bounds(**DESIRED_BOUNDS, transform=src.transform)
    sample_data = src.read(1, window=window)
    transform = rasterio.windows.transform(window, src.transform)
    bounds = rasterio.transform.array_bounds(*sample_data.shape, transform)

x = np.linspace(bounds[0], bounds[2], sample_data.shape[1])
y = np.linspace(bounds[1], bounds[3], sample_data.shape[0])

# Load all raster data
raster_stack = []
for path in RASTER_PATHS:
    with rasterio.open(path) as src:
        window = from_bounds(**DESIRED_BOUNDS, transform=src.transform)
        data = np.flipud(src.read(1, window=window))
        raster_stack.append(data)

#### Configuring Graphic

adding raster data to visualization

In [ ]:
# Create heatmap traces
traces = [
    go.Heatmap(
        z=data,
        x=x,
        y=y,
        visible=(i == 0),
        colorscale=[
            [0, 'rgb(255, 255, 204)'],  # Very light yellow (low values)
            [0.2, 'rgb(254, 217, 118)'],  # Soft yellow-orange
            [0.4, 'rgb(252, 141, 89)'],  # Light orange
            [0.6, 'rgb(227, 74, 51)'],   # Strong orange-red
            [0.8, 'rgb(179, 0, 0)'],     # Deep red
            [1, 'rgb(103, 0, 13)']       # Darkest red (high values)
        ],
        zmin=0,
        zmax=350,
        opacity=0.9,
        hoverinfo="none",
        showscale=True
    ) for i, data in enumerate(raster_stack)
]

In [ ]:
# Extract and format state boundaries from GeoJSON
if isinstance(filtered_geojson, str):
    filtered_geojson = json.loads(filtered_geojson)

latitudes = []
longitudes = []

for feature in filtered_geojson['features']:
    geom = shape(feature['geometry'])  # Convert GeoJSON geometry to Shapely object

    if geom.geom_type == "Polygon":
        coords = list(geom.exterior.coords)
        lon, lat = zip(*coords)
        longitudes.extend(lon + (None,))
        latitudes.extend(lat + (None,))

    elif geom.geom_type == "MultiPolygon":
        for polygon in geom.geoms:
            coords = list(polygon.exterior.coords)
            lon, lat = zip(*coords)
            longitudes.extend(lon + (None,))
            latitudes.extend(lat + (None,))

# Creating Visualization

In [ ]:
# initialize figure
fig = go.Figure()

# Add heatmap layers
for trace in traces:
    fig.add_trace(trace)

# Add state boundary lines (without tooltips)
fig.add_trace(go.Scatter(
    mode='lines',
    x=longitudes,
    y=latitudes,
    line=dict(color='rgba(0, 0, 0, 0.3)', width=1),
    name="State Boundaries",
    hoverinfo='skip'  # Explicitly disable tooltip
))

# Create slider
import calendar

steps = []
month_abbr = list(calendar.month_abbr)[1:]

for i in range(len(raster_stack)):
    step = dict(
        method='update',
        args=[{'visible': [False]*len(raster_stack) + [True]}],
        label=month_abbr[i % 12]  # Wraps around just in case
    )
    step['args'][0]['visible'][i] = True
    steps.append(step)

# Add slider to figure
fig.update_layout(
    sliders=[{
        'active': 0,
        'steps': steps,
        'x': 0.1,
        'len': 0.9
    }]
)

# Final layout settings
fig.update_layout(
    title='Average Monthly DNI 1998-2016',
    height=700,
    width=1150,
    margin=dict(l=50, r=50, t=100, b=100),
    geo=dict(
        scope="usa",
        projection_type="mercator"
    ),
    xaxis=dict(
        title="Longitude",
        range=[bounds[0], bounds[2]],
        showgrid=False
    ),
    yaxis=dict(
        title="Latitude",
        range=[bounds[1], bounds[3]],
        scaleanchor="x",
        showgrid=False
    ),
)

fig.show()

save image

In [ ]:
# fig.write_html(r"C:\Users\runni\OneDrive\Documents\UT\DViz Project\images\avg_monthly_DNI.html")